# 步驟 3：模型訓練與預測
運用 LightGBM 模型與 Walk-forward 模式避免偷看未來資料，並透過訊號濾網(Signal Filter)決定倉位大小。

In [ ]:
try:
    import lightgbm as lgb
except ImportError:
    print("❌ Install lightgbm first"); sys.exit(1)

LGBM_PARAMS = dict(
    n_estimators=800,
    max_depth=4,
    learning_rate=0.008,
    num_leaves=31,
    reg_alpha=0.1,
    reg_lambda=1.0,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

test_dates = [d for d in all_dates if d >= TEST_START]
print(f"  Test period : {test_dates[0].strftime('%Y-%m')} ~ {test_dates[-1].strftime('%Y-%m')} ({len(test_dates)} months)")

all_preds         = []
current_train_end = None
model             = None

for d in tqdm(test_dates, desc="  Walk-Forward"):
    # Retrain?
    if current_train_end is None or d >= current_train_end:
        train_start    = d - pd.DateOffset(months=TRAIN_MONTHS)
        purge_cutoff   = d - pd.DateOffset(months=PURGE_MONTHS)
        idx_tr = (
            (X.index.get_level_values(0) >= train_start) &
            (X.index.get_level_values(0) < purge_cutoff)
        )
        X_tr, y_tr = X[idx_tr], y[idx_tr]
        if len(X_tr) < 200:
            continue

        # Early-stopping on last 20% of train as val
        cut = int(len(X_tr) * 0.8)
        X_t_, y_t_ = X_tr.iloc[:cut], y_tr.iloc[:cut]
        X_v_, y_v_ = X_tr.iloc[cut:], y_tr.iloc[cut:]

        model = lgb.LGBMRegressor(**LGBM_PARAMS)
        model.fit(
            X_t_, y_t_,
            eval_set=[(X_v_, y_v_)],
            callbacks=[lgb.early_stopping(50, verbose=False),
                       lgb.log_evaluation(period=-1)],
        )
        current_train_end = d + pd.DateOffset(months=STEP_MONTHS)

    # Predict
    X_d = X[X.index.get_level_values(0) == d]
    if model is not None and not X_d.empty:
        preds_d = model.predict(X_d)
        all_preds.append(pd.DataFrame({"y_pred": preds_d}, index=X_d.index))

final_preds = pd.concat(all_preds)
print(f"\n  ✅ {len(final_preds):,} predictions — {len(tqdm._instances)} retrain cycles")

# Feature importance
fi = (pd.DataFrame({"Factor": X.columns, "Importance": model.feature_importances_})
        .sort_values("Importance", ascending=False))
print("\n  Feature Importance (last model):")
print(fi.to_string(index=False))

# ─── 6. Confidence-weighted positions ────────────────────────────────────────
print("\n[6/7] Building confidence-weighted positions…")

pred_wide = final_preds["y_pred"].unstack("stock_id")

# ══ Multi-signal Regime Filter ═══════════════════════════════════════════════
tw50 = close["0050"]
sig_60ma  = (tw50 > tw50.rolling(60).mean()).astype(float) * 0.4   # long-term trend
sig_20ma  = (tw50 > tw50.rolling(20).mean()).astype(float) * 0.3   # medium-term
sig_nodip = (tw50.pct_change(1) > -0.05).astype(float) * 0.3      # no big monthly drop
regime_score = sig_60ma + sig_20ma + sig_nodip
# Enter when 2 of 3 signals agree (score ≥ 0.4)
is_bull_daily   = regime_score >= 0.4
is_bull_monthly = (is_bull_daily.resample("ME").mean()
                                .reindex(pred_wide.index, method="ffill")
                                .fillna(0) >= 0.4)

bull_months = int(is_bull_monthly.sum())
print(f"  Bull months: {bull_months} / {len(is_bull_monthly)} "
      f"({bull_months/len(is_bull_monthly):.0%})  ← was 54% in v3")

# ══ Softmax confidence weighting ══════════════════════════════════════════════
def softmax_weights(scores: pd.Series, temp: float, top_k: int) -> pd.Series:
    """Select top_k stocks; weight by softmax(score / temp)."""
    top = scores.nlargest(top_k)
    exp_  = np.exp((top - top.max()) / temp)   # numerical stable
    return exp_ / exp_.sum()

weight_rows = {}
for date, row in pred_wide.iterrows():
    row = row.dropna()
    if is_bull_monthly.get(date, False) and len(row) >= TOP_K:
        weight_rows[date] = softmax_weights(row, WEIGHT_TEMP, TOP_K)
    else:
        weight_rows[date] = pd.Series(dtype=float)

# Wide weight table: rows=months, cols=stocks (0 for not held)
all_stocks   = pred_wide.columns
weights_df   = pd.DataFrame(weight_rows).T.reindex(columns=all_stocks).fillna(0.0)

active_months = (weights_df.sum(axis=1) > 0).sum()
print(f"  Active months: {active_months} / {len(weights_df)}")

# ─── 7. Vectorbt backtest ────────────────────────────────────────────────────